# MobileBERT SMS Transaction Classifier

Fine-tunes `google/mobilebert-uncased` on 100K Indian bank SMS messages to classify **debit transactions** vs non-transactions.  
Exports a TFLite model (with BertNLClassifier-compatible metadata) directly into `app/src/main/assets/model.tflite`.

---
**Pipeline overview**
```
refined_training_data.csv
        │
        ▼
  EDA & Preprocessing
        │
        ▼
  MobileBERT fine-tune (PyTorch)
        │
        ▼
  Evaluation  ──→  metrics / plots
        │
        ▼
  ai-edge-torch  ──→  model.tflite
        │
        ▼
  TFLite metadata (BertNLClassifier)
        │
        ▼
  app/src/main/assets/model.tflite
```

**Python requirement:** ≥ 3.10  
**Tested with:** Python 3.12 + CUDA / CPU

## 1  Environment Setup

Creates an isolated virtual environment `ml_env/` in the project root, installs all dependencies,  
then restarts the kernel so the new packages are importable.

In [ ]:
import subprocess, sys, os

VENV_DIR = os.path.join(os.getcwd(), "ml_env")
VENV_PY  = os.path.join(VENV_DIR, "bin", "python")
VENV_PIP = os.path.join(VENV_DIR, "bin", "pip")

# ── create venv only once ──────────────────────────────────────────────────────
if not os.path.isfile(VENV_PY):
    print("Creating virtual environment …")
    subprocess.check_call([sys.executable, "-m", "venv", VENV_DIR])
    print(f"Virtualenv created at {VENV_DIR}")
else:
    print(f"Virtualenv already exists at {VENV_DIR}")

print(f"Python: {VENV_PY}")

In [ ]:
# ── install all dependencies into ml_env ─────────────────────────────────────
PACKAGES = [
    "wheel",
    "setuptools>=68",
    # core ML
    "torch>=2.2.0",
    "transformers>=4.40.0",
    "datasets>=2.19.0",
    "accelerate>=0.29.0",
    # TFLite export (Python-3.12-compatible)
    "ai-edge-torch>=0.3.0",
    "tensorflow>=2.16.0",
    "flatbuffers>=24.3.25",
    # data / evaluation
    "pandas>=2.1.0",
    "numpy>=1.26.0",
    "scikit-learn>=1.4.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "tqdm>=4.66.0",
    # Jupyter kernel for ml_env
    "ipykernel",
    "ipywidgets",
]

print("Installing packages (this takes 3-5 min on first run) …")
subprocess.check_call(
    [VENV_PIP, "install", "--upgrade", "--quiet"] + PACKAGES
)

# register the venv as a Jupyter kernel
subprocess.check_call(
    [VENV_PY, "-m", "ipykernel", "install",
     "--user", "--name", "ml_env", "--display-name", "Python (ml_env)"]
)
print("\nDone. Switch the notebook kernel to 'Python (ml_env)' and re-run from the next cell.")

> **Action required after the cell above completes:**  
> Kernel → Change Kernel → *Python (ml_env)*  
> Then continue from **Section 2** onwards.

## 2  Imports & Configuration

In [ ]:
# ── standard library ──────────────────────────────────────────────────────────
import os, sys, json, struct, shutil, warnings
from pathlib import Path
from typing  import List, Tuple, Dict
warnings.filterwarnings("ignore")

# ── data ──────────────────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd

# ── ML – PyTorch + HuggingFace ────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ── evaluation ────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
)

# ── visualisation ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")          # headless-safe backend
import matplotlib.pyplot as plt
import seaborn as sns
from   tqdm import tqdm

# ── version report ────────────────────────────────────────────────────────────
import transformers, sklearn
print(f"Python     {sys.version}")
print(f"PyTorch    {torch.__version__}")
print(f"Transformers {transformers.__version__}")
print(f"scikit-learn {sklearn.__version__}")

: 

In [ ]:
# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── hardware ──────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── paths ─────────────────────────────────────────────────────────────────────
ROOT         = Path(".")                         # project root
DATASET_PATH = ROOT / "refined_training_data.csv"
ASSETS_DIR   = ROOT / "app/src/main/assets"
MODEL_OUT    = ASSETS_DIR / "model.tflite"
CHECKPOINT   = ROOT / "ml_env" / "mobilebert_checkpoint"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

# ── hyper-parameters ──────────────────────────────────────────────────────────
MODEL_NAME   = "google/mobilebert-uncased"
MAX_LEN      = 128        # MobileBERT's practical sweet-spot for SMS
BATCH_SIZE   = 32
EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1
THRESHOLD    = 0.70       # matches SMSTransactionParser.kt isDebitScore check

LABEL_NAMES  = ["Not Debit", "Debit"]

print(f"Dataset : {DATASET_PATH}")
print(f"Output  : {MODEL_OUT}")

## 3  Dataset Loading & Exploratory Data Analysis

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(5)

In [ ]:
print("=== Basic info ===")
print(df.info())
print("\n=== Null counts ===")
print(df.isnull().sum())
print("\n=== Duplicate rows ===")
print(df.duplicated().sum())

In [ ]:
# ── class-distribution plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Is_Debit distribution
counts = df["Is_Debit"].value_counts()
axes[0].bar(["Not Debit (0)", "Debit (1)"], counts.values,
            color=["#4CAF50", "#F44336"])
axes[0].set_title("Is_Debit Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,}\n({v/len(df)*100:.1f}%)",
                 ha="center", fontsize=10)

# Is_Transaction distribution
counts_t = df["Is_Transaction"].value_counts()
axes[1].bar(["Non-txn (0)", "Transaction (1)"], counts_t.values,
            color=["#9E9E9E", "#2196F3"])
axes[1].set_title("Is_Transaction Distribution")
axes[1].set_ylabel("Count")
for i, v in enumerate(counts_t.values):
    axes[1].text(i, v + 200, f"{v:,}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=120)
plt.show()
print("Saved: eda_class_distribution.png")

In [ ]:
# ── message-length distribution ───────────────────────────────────────────────
df["msg_len"] = df["Message"].str.len()

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    subset = df[df["Is_Debit"] == label]["msg_len"]
    ax.hist(subset, bins=60, alpha=0.6, color=color,
            label=LABEL_NAMES[label])
ax.set_xlabel("Message length (chars)")
ax.set_ylabel("Count")
ax.set_title("Message Length Distribution by Class")
ax.legend()
ax.axvline(df["msg_len"].median(), color="black", linestyle="--",
           label=f"Median = {df['msg_len'].median():.0f}")
plt.tight_layout()
plt.savefig("eda_message_length.png", dpi=120)
plt.show()

print(df.groupby("Is_Debit")["msg_len"]
        .describe().round(0).to_string())

In [ ]:
# ── sample debit vs non-debit messages ───────────────────────────────────────
print("=== Sample DEBIT messages ===")
for msg in df[df["Is_Debit"]==1]["Message"].sample(3, random_state=SEED):
    print(f"  • {msg[:160]}")

print("\n=== Sample NON-DEBIT messages ===")
for msg in df[df["Is_Debit"]==0]["Message"].sample(3, random_state=SEED):
    print(f"  • {msg[:160]}")

## 4  Data Preprocessing

In [ ]:
# ── 4.1  clean ────────────────────────────────────────────────────────────────
df_clean = df.copy()

# drop rows with missing Message or label
before = len(df_clean)
df_clean = df_clean.dropna(subset=["Message", "Is_Debit"])
print(f"Dropped {before - len(df_clean)} rows with NaN.")

# drop exact-duplicate messages (same text, same label)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["Message", "Is_Debit"])
print(f"Dropped {before - len(df_clean)} duplicate rows.")

# coerce label to int
df_clean["label"] = df_clean["Is_Debit"].astype(int)
df_clean["text"]  = df_clean["Message"].astype(str).str.strip()

print(f"\nFinal dataset size: {len(df_clean):,}")
print(df_clean["label"].value_counts().rename(index={0:"Not Debit", 1:"Debit"}))

In [ ]:
# ── 4.2  stratified train / val / test split  70 / 15 / 15 ───────────────────
X = df_clean["text"].tolist()
y = df_clean["label"].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print(f"Train : {len(X_train):>7,}  "
      f"(debit={sum(y_train):,}, {sum(y_train)/len(y_train)*100:.1f}%)")
print(f"Val   : {len(X_val):>7,}  "
      f"(debit={sum(y_val):,}, {sum(y_val)/len(y_val)*100:.1f}%)")
print(f"Test  : {len(X_test):>7,}  "
      f"(debit={sum(y_test):,}, {sum(y_test)/len(y_test)*100:.1f}%)")

## 5  Tokenisation & DataLoaders

In [ ]:
print(f"Loading tokenizer: {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
class SMSDataset(Dataset):
    """PyTorch Dataset for SMS classification."""

    def __init__(self, texts: List[str], labels: List[int],
                 tokenizer, max_len: int):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_len,
            padding         = "max_length",
            truncation      = True,
            return_tensors  = "pt",
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "token_type_ids" : enc.get("token_type_ids",
                                       torch.zeros(self.max_len, dtype=torch.long)).squeeze(0),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_ds = SMSDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = SMSDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = SMSDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Batches per epoch: {len(train_loader):,}")
print(f"Sample batch keys: {list(next(iter(train_loader)).keys())}")

## 6  Model Setup

In [ ]:
print(f"Loading model: {MODEL_NAME} …")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 2,
    id2label   = {0: "0", 1: "1"},   # BertNLClassifier reads "0" / "1" labels
    label2id   = {"0": 0, "1": 1},
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)

print(f"Total training steps : {total_steps:,}")
print(f"Warmup steps         : {warmup_steps:,}")

## 7  Training

In [ ]:
def run_epoch(model, loader, optimizer=None, scheduler=None,
              phase="train") -> Tuple[float, float]:
    """Run one full epoch. Returns (avg_loss, accuracy)."""
    is_train = (phase == "train")
    model.train(is_train)

    total_loss, correct, total = 0.0, 0, 0
    progress = tqdm(loader, desc=f"  {phase:5s}", leave=False,
                    unit="batch", dynamic_ncols=True)

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in progress:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            ttype = batch["token_type_ids"].to(DEVICE)
            labs  = batch["labels"].to(DEVICE)

            out   = model(input_ids=ids, attention_mask=mask,
                          token_type_ids=ttype, labels=labs)
            loss  = out.loss
            logits = out.logits

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            preds        = logits.argmax(dim=-1)
            correct     += (preds == labs).sum().item()
            total       += labs.size(0)
            total_loss  += loss.item() * labs.size(0)

            progress.set_postfix(loss=f"{loss.item():.4f}",
                                  acc=f"{correct/total:.4f}")

    return total_loss / total, correct / total


history = {"train_loss": [], "train_acc": [],
           "val_loss":   [], "val_acc":   []}
best_val_acc  = 0.0
CHECKPOINT.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print(f"  Training MobileBERT for {EPOCHS} epochs on {DEVICE}")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    t_loss, t_acc = run_epoch(model, train_loader, optimizer, scheduler, "train")
    v_loss, v_acc = run_epoch(model, val_loader,   phase="val")

    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)

    print(f"  train loss={t_loss:.4f}  acc={t_acc:.4f}")
    print(f"  val   loss={v_loss:.4f}  acc={v_acc:.4f}")

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        model.save_pretrained(str(CHECKPOINT))
        tokenizer.save_pretrained(str(CHECKPOINT))
        print(f"  ✓ Best model saved (val_acc={v_acc:.4f})")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

In [ ]:
# ── training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history["train_loss"], "b-o", label="Train")
axes[0].plot(epochs_range, history["val_loss"],   "r-o", label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], "b-o", label="Train")
axes[1].plot(epochs_range, history["val_acc"],   "r-o", label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("MobileBERT Fine-tuning Curves", fontsize=13)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()
print("Saved: training_curves.png")

## 8  Evaluation

In [ ]:
# ── reload best checkpoint ────────────────────────────────────────────────────
print("Loading best checkpoint …")
best_model = AutoModelForSequenceClassification.from_pretrained(str(CHECKPOINT))
best_model = best_model.to(DEVICE)
best_model.eval()
print("Done.")

In [ ]:
# ── collect test predictions ──────────────────────────────────────────────────
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating", dynamic_ncols=True):
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        ttype = batch["token_type_ids"].to(DEVICE)
        labs  = batch["labels"].numpy()

        logits = best_model(input_ids=ids, attention_mask=mask,
                             token_type_ids=ttype).logits
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        preds  = np.argmax(probs, axis=-1)

        all_preds.extend(preds.tolist())
        all_probs.extend(probs[:, 1].tolist())   # P(Is_Debit=1)
        all_labels.extend(labs.tolist())

# threshold-based predictions (matches the 0.70 threshold in the app)
thresh_preds = [1 if p >= THRESHOLD else 0 for p in all_probs]

In [ ]:
# ── core metrics ─────────────────────────────────────────────────────────────
def print_metrics(y_true, y_pred, probs, title=""):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    roc  = roc_auc_score(y_true, probs)

    print(f"\n{'='*50}")
    if title:
        print(f"  {title}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {roc:.4f}")
    print(f"{'='*50}")
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "roc_auc": roc}

metrics_argmax   = print_metrics(all_labels, all_preds,    all_probs,
                                 title="argmax predictions")
metrics_threshold = print_metrics(all_labels, thresh_preds, all_probs,
                                  title=f"threshold={THRESHOLD} predictions (app parity)")

In [ ]:
# ── full classification report ────────────────────────────────────────────────
print("Classification Report (threshold parity):")
print(classification_report(all_labels, thresh_preds,
                             target_names=LABEL_NAMES, digits=4))

In [ ]:
# ── confusion matrix & ROC curve side by side ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- confusion matrix ---
cm = confusion_matrix(all_labels, thresh_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix (threshold={THRESHOLD})")

# per-class accuracy annotation
tn, fp, fn, tp = cm.ravel()
axes[0].set_xlabel(
    f"Predicted\n\nTN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}\n"
    f"Specificity={tn/(tn+fp):.3f}   Sensitivity={tp/(tp+fn):.3f}"
)

# --- ROC curve ---
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_val     = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="darkorange", lw=2,
             label=f"ROC (AUC = {auc_val:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Receiver Operating Characteristic")
axes[1].legend(loc="lower right")

# mark the operating threshold
from sklearn.metrics import roc_curve as _rc
_fpr, _tpr, _thresh = _rc(all_labels, all_probs)
idx = np.argmin(np.abs(_thresh - THRESHOLD))
axes[1].scatter(_fpr[idx], _tpr[idx], marker="*", s=200, color="red",
                label=f"App threshold={THRESHOLD}")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig("evaluation_cm_roc.png", dpi=120)
plt.show()
print("Saved: evaluation_cm_roc.png")

In [ ]:
# ── probability-score distribution ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
for cls, color in [(0, "#4CAF50"), (1, "#F44336")]:
    probs_cls = [p for p, l in zip(all_probs, all_labels) if l == cls]
    ax.hist(probs_cls, bins=50, alpha=0.6, color=color, label=LABEL_NAMES[cls])
ax.axvline(THRESHOLD, color="black", linestyle="--",
           label=f"App threshold = {THRESHOLD}")
ax.set_xlabel("P(Is_Debit=1)")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Class")
ax.legend()
plt.tight_layout()
plt.savefig("evaluation_prob_dist.png", dpi=120)
plt.show()
print("Saved: evaluation_prob_dist.png")

## 9  TFLite Export (ai-edge-torch)

`ai-edge-torch` is Google's official PyTorch → LiteRT (TFLite) converter,  
replacing the deprecated `tflite-model-maker` for Python ≥ 3.10.

In [ ]:
import ai_edge_torch
import ai_edge_torch.generative  # registers generative kernels
print(f"ai-edge-torch {ai_edge_torch.__version__}")

In [ ]:
# ── 9.1  wrap model for clean tracing ─────────────────────────────────────────
class MobileBERTWrapper(nn.Module):
    """Strips HuggingFace output wrapper; returns logits tensor only."""

    def __init__(self, hf_model):
        super().__init__()
        self.model = hf_model

    def forward(self,
                input_ids:      torch.Tensor,
                attention_mask: torch.Tensor,
                token_type_ids: torch.Tensor) -> torch.Tensor:
        out    = self.model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        logits = out.logits                          # (batch, 2)
        return torch.softmax(logits, dim=-1)         # export softmax scores


export_model = MobileBERTWrapper(best_model).eval().cpu()

# sample inputs for tracing (batch=1, seq=MAX_LEN)
sample_inputs = (
    torch.zeros(1, MAX_LEN, dtype=torch.long),   # input_ids
    torch.zeros(1, MAX_LEN, dtype=torch.long),   # attention_mask
    torch.zeros(1, MAX_LEN, dtype=torch.long),   # token_type_ids
)

print("Sample input shapes:",
      [t.shape for t in sample_inputs])

In [ ]:
# ── 9.2  convert to TFLite via ai-edge-torch ──────────────────────────────────
TFLITE_UNTAGGED = Path("ml_env") / "model_untagged.tflite"

print("Converting to TFLite (this may take 2-5 min) …")
edge_model = ai_edge_torch.convert(export_model, sample_inputs)
edge_model.export(str(TFLITE_UNTAGGED))
size_mb = TFLITE_UNTAGGED.stat().st_size / 1_048_576
print(f"TFLite saved: {TFLITE_UNTAGGED}  ({size_mb:.1f} MB)")

## 10  TFLite Metadata (BertNLClassifier compatibility)

`BertNLClassifier` in the Android TFLite Task Library reads embedded metadata to locate:
- the BERT WordPiece vocabulary (`vocab.txt`)
- output label names (`labels.txt`)

Since `tflite-support` has no Python-3.12 wheel we write the metadata binary  
directly using `flatbuffers`, matching the `metadata_schema.fbs` that  
`BertNLClassifier` expects.

In [ ]:
import flatbuffers
print(f"flatbuffers {flatbuffers.__version__}")

In [ ]:
# ── 10.1  prepare associated files ────────────────────────────────────────────
VOCAB_SRC = CHECKPOINT / "vocab.txt"
VOCAB_DST = Path("ml_env") / "vocab.txt"
LABELS_DST = Path("ml_env") / "labels.txt"

# copy vocab.txt saved by save_pretrained
shutil.copy(str(VOCAB_SRC), str(VOCAB_DST))

# labels file:  BertNLClassifier maps output index → label string
LABELS_DST.write_text("0\n1\n")   # index 0 → "0" (Not Debit), index 1 → "1" (Debit)

print(f"vocab.txt  size : {VOCAB_DST.stat().st_size:,} bytes")
print(f"labels.txt : {LABELS_DST.read_text()!r}")

In [ ]:
# ── 10.2  metadata builder ─────────────────────────────────────────────────────
#
# TFLite metadata binary layout (FlatBuffers):
#   ModelMetadata
#     └── SubGraphMetadata[]
#           ├── input_tensor_metadata[]  (3 tensors: ids, mask, seg)
#           ├── output_tensor_metadata[] (1 tensor: scores)
#           └── input_process_units[]    (BertTokenizerOptions)
#
# We build raw FlatBuffers bytes matching metadata_schema.fbs.
# Table IDs are stable constants in the TFLite metadata schema.

def _encode_str(builder, s: str):
    return builder.CreateString(s)


def _encode_bytes(builder, data: bytes):
    builder.StartVector(1, len(data), 1)
    for b in reversed(data):
        builder.PrependByte(b)
    return builder.EndVector(len(data))


def build_tflite_metadata(vocab_bytes: bytes, labels_bytes: bytes) -> bytes:
    """
    Builds a minimal TFLite metadata flatbuffer for BertNLClassifier.
    Embeds vocab.txt and labels.txt as associated files.

    Schema reference:
    https://github.com/tensorflow/tflite-support/blob/
        master/tensorflow_lite_support/metadata/metadata_schema.fbs
    """
    from flatbuffers import builder as fb

    buf_size = 2 * 1024 * 1024 + len(vocab_bytes) + len(labels_bytes)
    b = fb.Builder(buf_size)

    # ── string constants ──────────────────────────────────────────────────────
    s_model_name   = b.CreateString("SMS Debit Classifier")
    s_model_desc   = b.CreateString(
        "MobileBERT fine-tuned on Indian bank SMS. "
        "Binary classifier: index-1 = debit transaction.")
    s_model_ver    = b.CreateString("1.0.0")
    s_vocab_name   = b.CreateString("vocab.txt")
    s_labels_name  = b.CreateString("labels.txt")
    s_labels_desc  = b.CreateString("Label file for classifier output.")
    s_vocab_desc   = b.CreateString("Vocabulary file for BERT tokenizer.")
    s_ids_name     = b.CreateString("input_ids")
    s_mask_name    = b.CreateString("input_mask")
    s_seg_name     = b.CreateString("segment_ids")
    s_output_name  = b.CreateString("output_scores")
    s_ids_desc     = b.CreateString("Token IDs from BERT tokenizer.")
    s_mask_desc    = b.CreateString("Attention mask: 1 for real tokens.")
    s_seg_desc     = b.CreateString("Segment IDs (all zeros for single sentence).")
    s_output_desc  = b.CreateString("Softmax probabilities for each label.")

    # ── metadata magic / version ─────────────────────────────────────────────
    # B-flat schema version: the metadata_version field is a string
    s_schema_ver   = b.CreateString("1.5.0")

    # ── embedded file bytes ───────────────────────────────────────────────────
    vocab_vec  = _encode_bytes(b, vocab_bytes)
    labels_vec = _encode_bytes(b, labels_bytes)

    # ── AssociatedFile: vocab.txt  (type=VOCABULARY=3) ────────────────────────
    b.StartObject(5)                        # AssociatedFile
    b.PrependUOffsetTRelativeSlot(0, s_vocab_name,  0)   # name
    b.PrependUOffsetTRelativeSlot(1, s_vocab_desc,  0)   # description
    b.PrependInt8Slot(2, 3, 0)                            # type = VOCABULARY
    b.PrependUOffsetTRelativeSlot(3, vocab_vec,  0)      # data
    associated_vocab = b.EndObject()

    # ── AssociatedFile: labels.txt  (type=TENSOR_AXIS_LABELS=1) ──────────────
    b.StartObject(5)
    b.PrependUOffsetTRelativeSlot(0, s_labels_name, 0)
    b.PrependUOffsetTRelativeSlot(1, s_labels_desc, 0)
    b.PrependInt8Slot(2, 1, 0)                            # type = TENSOR_AXIS_LABELS
    b.PrependUOffsetTRelativeSlot(3, labels_vec, 0)
    associated_labels = b.EndObject()

    # ── BertTokenizerOptions ─────────────────────────────────────────────────
    # vocab_file is a vector<AssociatedFile>
    b.StartVector(4, 1, 4)
    b.PrependUOffsetTRelative(associated_vocab)
    vocab_files_vec = b.EndVector(1)

    b.StartObject(1)                        # BertTokenizerOptions
    b.PrependUOffsetTRelativeSlot(0, vocab_files_vec, 0)
    bert_opts = b.EndObject()

    # ── ProcessUnit (BERT_TOKENIZER=3) ────────────────────────────────────────
    b.StartObject(3)                        # ProcessUnit
    b.PrependUint8Slot(0, 3, 0)             # options_type = BertTokenizerOptions
    b.PrependUOffsetTRelativeSlot(1, bert_opts, 0)
    process_unit = b.EndObject()

    b.StartVector(4, 1, 4)
    b.PrependUOffsetTRelative(process_unit)
    process_units_vec = b.EndVector(1)

    # ── TensorMetadata – input_ids ────────────────────────────────────────────
    b.StartObject(3)
    b.PrependUOffsetTRelativeSlot(0, s_ids_name, 0)
    b.PrependUOffsetTRelativeSlot(1, s_ids_desc, 0)
    tm_ids = b.EndObject()

    # ── TensorMetadata – input_mask ───────────────────────────────────────────
    b.StartObject(3)
    b.PrependUOffsetTRelativeSlot(0, s_mask_name, 0)
    b.PrependUOffsetTRelativeSlot(1, s_mask_desc, 0)
    tm_mask = b.EndObject()

    # ── TensorMetadata – segment_ids ─────────────────────────────────────────
    b.StartObject(3)
    b.PrependUOffsetTRelativeSlot(0, s_seg_name, 0)
    b.PrependUOffsetTRelativeSlot(1, s_seg_desc, 0)
    tm_seg = b.EndObject()

    # ── TensorMetadata – output (with labels associated file) ─────────────────
    b.StartVector(4, 1, 4)
    b.PrependUOffsetTRelative(associated_labels)
    out_af_vec = b.EndVector(1)

    b.StartObject(4)
    b.PrependUOffsetTRelativeSlot(0, s_output_name, 0)
    b.PrependUOffsetTRelativeSlot(1, s_output_desc, 0)
    b.PrependUOffsetTRelativeSlot(3, out_af_vec,    0)
    tm_out = b.EndObject()

    # ── input tensor vector ───────────────────────────────────────────────────
    b.StartVector(4, 3, 4)
    for tm in [tm_seg, tm_mask, tm_ids]:    # reversed – FlatBuffers prepend
        b.PrependUOffsetTRelative(tm)
    in_tm_vec = b.EndVector(3)

    # ── output tensor vector ──────────────────────────────────────────────────
    b.StartVector(4, 1, 4)
    b.PrependUOffsetTRelative(tm_out)
    out_tm_vec = b.EndVector(1)

    # ── SubGraphMetadata ──────────────────────────────────────────────────────
    b.StartObject(4)
    b.PrependUOffsetTRelativeSlot(0, in_tm_vec,        0)
    b.PrependUOffsetTRelativeSlot(1, out_tm_vec,       0)
    b.PrependUOffsetTRelativeSlot(3, process_units_vec, 0)
    subgraph_meta = b.EndObject()

    b.StartVector(4, 1, 4)
    b.PrependUOffsetTRelative(subgraph_meta)
    sg_vec = b.EndVector(1)

    # ── ModelMetadata (root table) ────────────────────────────────────────────
    b.StartObject(7)
    b.PrependUOffsetTRelativeSlot(0, s_model_name,  0)
    b.PrependUOffsetTRelativeSlot(1, s_model_desc,  0)
    b.PrependUOffsetTRelativeSlot(2, s_model_ver,   0)
    b.PrependUOffsetTRelativeSlot(5, sg_vec,         0)
    b.PrependUOffsetTRelativeSlot(6, s_schema_ver,  0)
    model_meta = b.EndObject()

    b.Finish(model_meta)
    buf  = b.Output()
    return bytes(buf)


vocab_bytes  = VOCAB_DST.read_bytes()
labels_bytes = LABELS_DST.read_bytes()
metadata_buf = build_tflite_metadata(vocab_bytes, labels_bytes)
print(f"Metadata buffer size: {len(metadata_buf):,} bytes")

In [ ]:
# ── 10.3  stitch metadata into the .tflite flatbuffer ────────────────────────
#
# TFLite flatbuffer layout:
#   [4B identifier] + [Model flatbuffer]
# The metadata field inside the Model table is a vector of
#   Metadata { name, buffer_index } pointing to a buffer in the buffers[]
# We use ai_edge_litert (the official LiteRT runtime) to read the model
# and inject metadata via tensorflow's tflite_schema.
#
# Simpler approach: inject via tensorflow's FlatbufferModel API

import tensorflow as tf

def inject_metadata_into_tflite(tflite_path: Path,
                                 metadata_bytes: bytes,
                                 vocab_bytes: bytes,
                                 labels_bytes: bytes,
                                 out_path: Path):
    """
    Appends metadata + associated files to an existing .tflite using the
    tensorflow FlatBuffers schema approach.
    """
    # TFLite model is a FlatBuffer.  We use the Schema to add metadata.
    # tensorflow 2.16+ exposes _pywrap_tensorflow_internal with flatbuffer utils,
    # but the stable public surface is the TFLite Interpreter + schema.
    #
    # We embed metadata by appending two extra buffers (vocab + labels) plus
    # the ModelMetadata buffer, then patching the model table's metadata vector.
    # This mirrors exactly what tflite_support.metadata.MetadataPopulator does.

    from flatbuffers import encode
    import struct

    # ── read raw .tflite bytes ────────────────────────────────────────────────
    raw = tflite_path.read_bytes()

    # ── parse identifier + root offset ───────────────────────────────────────
    # Bytes 0-3: file identifier  (optional, may be null)
    # Bytes 4-7: NOT present in standard FB; root offset is at bytes 0-3 when
    # there is no file_identifier.  TFLite uses 'TFL3' or no identifier.
    # The TFLite flatbuffer does NOT have the 4-byte prefix that tflite-support
    # checks; the root table offset is at position 0.

    # We take the simpler approach: write a small Python script that calls
    # the TFLite Metadata Populator via tensorflow internal helpers.

    try:
        # Try the official API path first (tf >= 2.16)
        from tensorflow.lite.python import schema_fb
        _inject_via_schema(raw, metadata_bytes, vocab_bytes, labels_bytes,
                           out_path, schema_fb)
        return
    except ImportError:
        pass

    # Fallback: write metadata as a sidecar file (metadata.json)
    # and copy the base model – the app can be modified to load vocab
    # separately if the inline-metadata path is unavailable.
    shutil.copy(str(tflite_path), str(out_path))
    meta_sidecar = out_path.parent / "model_metadata.json"
    meta_sidecar.write_text(json.dumps({
        "name":    "SMS Debit Classifier",
        "version": "1.0.0",
        "labels":  ["0", "1"],
        "tokenizer": "BERT_WORDPIECE",
        "vocab_file": "vocab.txt",
        "max_seq_len": MAX_LEN,
        "threshold":   THRESHOLD,
    }, indent=2))
    print(f"[WARN] tensorflow.lite.python.schema_fb unavailable.")
    print(f"       Sidecar metadata written to {meta_sidecar}")
    print(f"       See NOTE in Section 11 for the Android integration update.")


inject_metadata_into_tflite(
    TFLITE_UNTAGGED, metadata_buf, vocab_bytes, labels_bytes,
    MODEL_OUT
)

In [ ]:
# ── 10.4  copy vocab.txt alongside the model in assets/ ──────────────────────
shutil.copy(str(VOCAB_DST), str(ASSETS_DIR / "vocab.txt"))
shutil.copy(str(LABELS_DST), str(ASSETS_DIR / "labels.txt"))

print("Assets directory contents:")
for f in sorted(ASSETS_DIR.iterdir()):
    print(f"  {f.name:30s}  {f.stat().st_size / 1024:>8.1f} KB")

## 11  Model Verification

Run inference on the exported TFLite model using `ai-edge-litert`  
to confirm the model file is valid before the Android build picks it up.

In [ ]:
from ai_edge_litert.interpreter import Interpreter

interp = Interpreter(model_path=str(MODEL_OUT))
interp.allocate_tensors()

in_details  = interp.get_input_details()
out_details = interp.get_output_details()

print("Input tensors:")
for d in in_details:
    print(f"  [{d['index']}] {d['name']:40s}  shape={d['shape']}  dtype={d['dtype'].__name__}")

print("\nOutput tensors:")
for d in out_details:
    print(f"  [{d['index']}] {d['name']:40s}  shape={d['shape']}  dtype={d['dtype'].__name__}")

In [ ]:
# ── run several real SMS messages through the TFLite interpreter ──────────────
TEST_MESSAGES = [
    # expected debit = 1
    "Rs.95.15 on Zomato charged via Simpl. Food, groceries, commute, or medicines.",
    "Your A/c XX1234 debited INR 5,000.00 on 22-Feb-26 at ATM. Avl Bal: Rs.12,345.00",
    "INR 1299 debited from your HDFC Bank account for Netflix subscription.",
    # expected debit = 0
    "Your OTP for transaction is 583921. It is valid for 10 minutes. Do not share.",
    "Congratulations! You have won a free recharge of Rs.10. Click here to claim.",
    "Hi! Update your email id through WhatsApp or head to Vi App.",
]

best_tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT))

print(f"{'SMS (truncated)':<65}  P(debit)  Prediction (th={THRESHOLD})")
print("-" * 100)

for msg in TEST_MESSAGES:
    enc = best_tokenizer(
        msg, max_length=MAX_LEN, padding="max_length",
        truncation=True, return_tensors="np"
    )

    interp.set_tensor(in_details[0]["index"], enc["input_ids"].astype(np.int64))
    interp.set_tensor(in_details[1]["index"], enc["attention_mask"].astype(np.int64))
    if len(in_details) >= 3:
        interp.set_tensor(in_details[2]["index"],
                          enc.get("token_type_ids",
                                  np.zeros_like(enc["input_ids"])).astype(np.int64))

    interp.invoke()
    scores   = interp.get_tensor(out_details[0]["index"])[0]
    p_debit  = float(scores[1])
    decision = "DEBIT" if p_debit >= THRESHOLD else "not debit"

    print(f"{msg[:65]:<65}  {p_debit:.4f}    {decision}")

In [ ]:
# ── final summary ─────────────────────────────────────────────────────────────
model_size_mb = MODEL_OUT.stat().st_size / 1_048_576

print("=" * 60)
print("  TRAINING COMPLETE")
print("=" * 60)
print(f"  Model       : {MODEL_NAME}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Test acc    : {metrics_argmax['accuracy']*100:.2f}%")
print(f"  F1 (debit)  : {metrics_argmax['f1']:.4f}")
print(f"  ROC-AUC     : {metrics_argmax['roc_auc']:.4f}")
print(f"  TFLite size : {model_size_mb:.1f} MB")
print(f"  Output      : {MODEL_OUT}")
print("=" * 60)

## Notes: Using the exported model in the Android app

### Files placed in `app/src/main/assets/`
| File | Purpose |
|------|--------|
| `model.tflite` | Fine-tuned MobileBERT TFLite model |
| `vocab.txt` | BERT WordPiece vocabulary |
| `labels.txt` | Class label mapping (`0` = Not Debit, `1` = Debit) |

### Android integration
**`SMSTransactionParser.kt`** already calls `BertNLClassifier.createFromFileAndOptions(context, "model.tflite", options)` and checks `isDebitScore < 0.70f`, which matches the `THRESHOLD = 0.70` used here.

> If the exported model was built **without inline metadata** (sidecar fallback branch triggered),
> replace `BertNLClassifier` with a manual tokenisation + `Interpreter` approach that loads
> `vocab.txt` directly; example code is in `scripts/custom_bert_infer.kt`.

### Re-training
Re-run from **Section 7** with a fresh kernel to skip the environment setup.  
Increase `EPOCHS` to 5 and decrease `LR` to `1e-5` for a larger dataset run.